# Phishing Website Detection
Trains a Logistic Regression classifier on URL text features to detect phishing sites.

## 1. Imports

In [ ]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from nltk.tokenize import RegexpTokenizer
from nltk.stem.snowball import SnowballStemmer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

%matplotlib inline

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('Dataset/phishing_site_urls.csv')

print('Shape:', df.shape)
print()
df.info()
print()
print('Missing values:\n', df.isnull().sum())
print()
print('Label distribution:\n', df['Label'].value_counts())

## 3. Feature Engineering
Tokenize and stem each URL, then join tokens into a single text string per sample.

In [ ]:
tokenizer = RegexpTokenizer(r'[A-Za-z]+')
stemmer = SnowballStemmer('english')

df['text_tokenized'] = df['URL'].map(lambda t: tokenizer.tokenize(t))
df['text_stemmed']   = df['text_tokenized'].map(lambda l: [stemmer.stem(w) for w in l])
df['text']           = df['text_stemmed'].map(lambda l: ' '.join(l))

df.head()

## 4. Word Cloud Visualisation

In [ ]:
good_sites = df[df['Label'] == 'good']
bad_sites  = df[df['Label'] == 'bad']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, subset, title in [
    (axes[0], good_sites, 'Legitimate URLs'),
    (axes[1], bad_sites,  'Phishing URLs'),
]:
    text = ' '.join(subset['text'].tolist())
    wc = WordCloud(width=800, height=400, background_color='white').generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14)

plt.tight_layout()
plt.show()

## 5. Vectorisation & Train / Test Split

In [ ]:
cv = CountVectorizer()
features = cv.fit_transform(df['text'])

x_train, x_test, y_train, y_test = train_test_split(
    features, df['Label'], test_size=0.2, random_state=42
)

print(f'Train samples: {x_train.shape[0]}  |  Test samples: {x_test.shape[0]}')

## 6. Model Training & Evaluation

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

print(f'Test accuracy:  {model.score(x_test, y_test):.4f}')
print(f'Train accuracy: {model.score(x_train, y_train):.4f}')

In [ ]:
print('Classification Report\n')
print(classification_report(y_test, model.predict(x_test), target_names=['Bad', 'Good']))

In [ ]:
cm = confusion_matrix(y_test, model.predict(x_test))
cm_df = pd.DataFrame(
    cm,
    columns=['Predicted: Bad', 'Predicted: Good'],
    index=['Actual: Bad', 'Actual: Good']
)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Save Model & Vectorizer

In [ ]:
pickle.dump(model, open('phishing.pkl', 'wb'))
pickle.dump(cv,    open('vectorizer.pkl', 'wb'))
print('Saved: phishing.pkl, vectorizer.pkl')